<a href="https://colab.research.google.com/github/eodenyire/LearnDataScienceWithEmmanuelOdenyire/blob/main/Phase_1_Foundations_Python_and_Math/Month_03_Data_Visualization/Week_4_Plotly_and_Interactive/Day_27_Dashboards_with_Plotly_Dash.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="https://github.com/user-attachments/assets/2a4adf95-3d28-41e2-90d0-d06b2e9c8fa3" alt="Learn Data Science with Emmanuel Odenyire" width="180"/>

# Day 27: Dashboards with Plotly Dash

## Phase 1 | Month 3 | Week 4 | Day 27

**🎯 Goal:** Build a multi-chart web dashboard using Plotly Dash with callbacks for real-time interactivity.
**⏱️ Study Session:** ~2 hours of this 15-hour day

---

### What is Dash?
Dash is a Python framework for building analytical web applications — without JavaScript. It runs in your browser and updates charts reactively when users interact with dropdowns, sliders, and date pickers.

### Dash vs Streamlit vs Panel

| Feature | Dash | Streamlit | Panel |
|---------|------|-----------|-------|
| Learning curve | Medium | Low | Medium |
| Customisation | High | Medium | High |
| Production use | Yes | Yes | Yes |
| Callbacks | Explicit | Implicit | Explicit |

**In industry:** Safaricom BI team uses Dash; startups often use Streamlit for speed.

In [ ]:
# Install Dash (uncomment in Colab or local environment)
# !pip install dash

import dash
from dash import dcc, html, Input, Output
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import numpy as np

print(f'Dash version: {dash.__version__}')

In [ ]:
import dash
from dash import dcc, html, Input, Output
import plotly.express as px
import pandas as pd
import numpy as np

np.random.seed(42)
n = 500
df = pd.DataFrame({
    'country'    : np.random.choice(['Kenya','Nigeria','Ghana','Ethiopia','Tanzania','Rwanda'], n),
    'year'       : np.random.choice(list(range(2015, 2024)), n),
    'gdp_pc'     : np.random.lognormal(7.5, 0.7, n),
    'internet_pct': np.random.beta(3, 4, n) * 100,
    'mobile_pct' : np.random.beta(4, 3, n) * 100,
    'literacy'   : np.random.beta(5, 2, n) * 100,
    'population' : np.random.uniform(10, 230, n),
})

# ── Simple Dash App Layout ────────────────────────────────────────────────────
app = dash.Dash(__name__)

app.layout = html.Div([
    html.H1('African Development Dashboard',
            style={'textAlign': 'center', 'color': '#2C3E50', 'fontFamily': 'Arial'}),

    html.Div([
        html.Div([
            html.Label('Select X-Axis:', style={'fontWeight': 'bold'}),
            dcc.Dropdown(
                id='x-axis',
                options=[{'label': 'GDP per Capita', 'value': 'gdp_pc'},
                         {'label': 'Internet Penetration (%)', 'value': 'internet_pct'},
                         {'label': 'Mobile Penetration (%)', 'value': 'mobile_pct'},
                         {'label': 'Literacy Rate (%)', 'value': 'literacy'}],
                value='gdp_pc',
                clearable=False
            ),
        ], style={'width': '45%', 'display': 'inline-block', 'padding': '10px'}),

        html.Div([
            html.Label('Select Y-Axis:', style={'fontWeight': 'bold'}),
            dcc.Dropdown(
                id='y-axis',
                options=[{'label': 'GDP per Capita', 'value': 'gdp_pc'},
                         {'label': 'Internet Penetration (%)', 'value': 'internet_pct'},
                         {'label': 'Mobile Penetration (%)', 'value': 'mobile_pct'},
                         {'label': 'Literacy Rate (%)', 'value': 'literacy'}],
                value='internet_pct',
                clearable=False
            ),
        ], style={'width': '45%', 'display': 'inline-block', 'padding': '10px'}),
    ]),

    html.Div([
        html.Label('Filter by Year:', style={'fontWeight': 'bold', 'padding': '10px'}),
        dcc.Slider(
            id='year-slider',
            min=2015, max=2023, step=1, value=2023,
            marks={yr: str(yr) for yr in range(2015, 2024)},
            tooltip={'placement': 'bottom', 'always_visible': True}
        ),
    ], style={'padding': '20px'}),

    dcc.Graph(id='main-scatter'),

], style={'backgroundColor': '#F8F9FA', 'padding': '20px', 'fontFamily': 'Arial'})

@app.callback(
    Output('main-scatter', 'figure'),
    Input('x-axis', 'value'),
    Input('y-axis', 'value'),
    Input('year-slider', 'value')
)
def update_scatter(x_col, y_col, selected_year):
    filtered = df[df['year'] == selected_year]
    fig = px.scatter(
        filtered, x=x_col, y=y_col,
        color='country', size='population',
        hover_name='country',
        log_x=(x_col == 'gdp_pc'),
        title=f'{x_col.replace("_"," ").title()} vs {y_col.replace("_"," ").title()} ({selected_year})',
        template='plotly_white', height=500
    )
    return fig

print('Dash app defined. Run with: app.run(debug=True, port=8050)')
print('In Jupyter: app.run(jupyter_mode="inline")')
print()
print('App layout:')
print('  - Dropdown: X-axis selection')
print('  - Dropdown: Y-axis selection')
print('  - Slider: Year filter')
print('  - Scatter chart: updates on every interaction')

In [ ]:
# ── Dash app with multiple chart types ───────────────────────────────────────
import dash
from dash import dcc, html, Input, Output
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np

np.random.seed(42)
n = 500
df = pd.DataFrame({
    'country'    : np.random.choice(['Kenya','Nigeria','Ghana','Ethiopia','Tanzania','Rwanda'], n),
    'year'       : np.random.choice(list(range(2015, 2024)), n),
    'gdp_pc'     : np.random.lognormal(7.5, 0.7, n),
    'internet_pct': np.random.beta(3, 4, n) * 100,
    'population' : np.random.uniform(10, 230, n),
})

print('Dash Callback Pattern:')
print('''
@app.callback(
    Output(component_id="chart-id", component_property="figure"),
    Input(component_id="dropdown-id",  component_property="value"),
    Input(component_id="slider-id",    component_property="value")
)
def update_chart(dropdown_value, slider_value):
    # Filter data based on inputs
    filtered_df = df[df["year"] == slider_value]
    # Create figure
    fig = px.scatter(filtered_df, x="gdp_pc", y="internet_pct")
    return fig
''')

print('Key Dash Components:')
components = [
    ('dcc.Dropdown', 'Select from options'),
    ('dcc.Slider', 'Numeric range selector'),
    ('dcc.DatePickerRange', 'Date range selector'),
    ('dcc.Checklist', 'Multi-select checkboxes'),
    ('dcc.RadioItems', 'Single-select radio buttons'),
    ('dcc.Input', 'Text/number input'),
    ('dcc.Graph', 'Plotly chart container'),
    ('html.Div', 'Layout container'),
    ('html.H1-H6', 'Headings'),
    ('html.Button', 'Clickable button'),
]
for comp, desc in components:
    print(f'  {comp:25} — {desc}')

## 💪 Exercises

### Exercise 1
Add a **CheckList** component to the Dash app above to filter by country (allow multiple country selection). Update the scatter plot to only show selected countries.

### Exercise 2
Add a second graph to the Dash app showing the histogram of the selected X-axis variable, filtered by the same year.

In [ ]:
import dash
from dash import dcc, html, Input, Output
import plotly.express as px
import numpy as np, pandas as pd
np.random.seed(42)

# YOUR CODE HERE — extend the app above


## 📋 Summary

- ✓ Dash layout with `html.Div`, `dcc.Dropdown`, `dcc.Slider`, `dcc.Graph`
- ✓ `@app.callback()` decorator connects inputs to outputs reactively
- ✓ `app.run(debug=True)` to start the server
- ✓ One Python file = a full interactive web dashboard

## 🚀 Next Steps
**Day 28: Month 3 Final Review and Capstone Project** — build a complete interactive data story!